# 00 — Mount ADLS Gen2 Storage
**Day 1 | Part 7**

Mounts all 4 containers (`bronze`, `silver`, `gold`, `source`) from your ADLS Gen2 storage account into Databricks using Service Principal OAuth.

**Run this notebook once per cluster restart** — mounts are lost when a cluster terminates.

**Prerequisites:**
- Key Vault secret scope `kv-ev-scope` created (Day 1 Part 6.5)
- All secrets added to Key Vault (Day 1 Part 4.3 + Part 5)
- Cluster Access mode = **Dedicated** (not Standard/Shared)

## Cell 1 — Verify secret scope exists before mounting

In [ ]:
# Verify the secret scope is accessible before attempting to mount
print("Checking secret scope...")
try:
    scopes = [s.name for s in dbutils.secrets.listScopes()]
    if "kv-ev-scope" not in scopes:
        raise Exception("Secret scope 'kv-ev-scope' not found. Create it first via Day 1 Part 6.5.")
    print(f"  kv-ev-scope found — OK")
    print(f"  All scopes available: {scopes}")
except Exception as e:
    print(f"  ERROR: {e}")
    raise

## Cell 2 — Load all secrets from Key Vault

In [ ]:
SCOPE = "kv-ev-scope"

try:
    storage_account  = dbutils.secrets.get(scope=SCOPE, key="adls-account-name")
    sp_client_id     = dbutils.secrets.get(scope=SCOPE, key="sp-client-id")
    sp_client_secret = dbutils.secrets.get(scope=SCOPE, key="sp-client-secret")
    sp_tenant_id     = dbutils.secrets.get(scope=SCOPE, key="sp-tenant-id")
    print(f"Storage account : {storage_account}")
    print(f"SP client ID    : {sp_client_id[:8]}...[REDACTED]")
    print(f"SP tenant ID    : {sp_tenant_id}")
    print("All secrets loaded — OK")
except Exception as e:
    print(f"ERROR loading secrets: {e}")
    print("Check that all secrets exist in Key Vault:")
    print("  adls-account-name, sp-client-id, sp-client-secret, sp-tenant-id")
    raise

## Cell 3 — Mount all 4 containers using Service Principal OAuth

In [ ]:
# OAuth config — Databricks exchanges client_id + client_secret for a short-lived token
configs = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type":
        "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": sp_client_id,
    "fs.azure.account.oauth2.client.secret": sp_client_secret,
    "fs.azure.account.oauth2.client.endpoint":
        f"https://login.microsoftonline.com/{sp_tenant_id}/oauth2/token",
}

containers = ["bronze", "silver", "gold", "source"]
mounted, already, failed = [], [], []

for container in containers:
    mount_point = f"/mnt/{container}"
    try:
        if any(m.mountPoint == mount_point for m in dbutils.fs.mounts()):
            print(f"  Already mounted : {mount_point}")
            already.append(container)
        else:
            dbutils.fs.mount(
                source=f"abfss://{container}@{storage_account}.dfs.core.windows.net/",
                mount_point=mount_point,
                extra_configs=configs,
            )
            print(f"  Mounted         : {mount_point}")
            mounted.append(container)
    except Exception as e:
        print(f"  FAILED          : {mount_point} — {e}")
        failed.append(container)

print(f"\nSummary:")
print(f"  Newly mounted   : {mounted}")
print(f"  Already mounted : {already}")
print(f"  Failed          : {failed}")

if failed:
    print("\nFor 403 errors: SP does not have Storage Blob Data Contributor role on the storage account.")
    print("For 'invalid client secret': check sp-client-secret value in Key Vault.")
    raise Exception(f"Mount failed for: {failed}")

## Cell 4 — Verify all mounts are accessible

In [ ]:
print("=== Verifying all mounts ===\n")
all_ok = True

for container in ["bronze", "silver", "gold", "source"]:
    mount_point = f"/mnt/{container}"
    try:
        items = dbutils.fs.ls(mount_point)
        print(f"  {mount_point:<20} OK — {len(items)} items")
    except Exception as e:
        print(f"  {mount_point:<20} ERROR — {e}")
        all_ok = False

if all_ok:
    print("\nAll 4 mounts verified — storage is accessible.")
else:
    print("\nSome mounts failed verification. Re-run Cell 3 after fixing the error.")

## Cell 5 — Show all current mounts (reference)

In [ ]:
print("All /mnt/ mounts on this cluster:")
for m in sorted(dbutils.fs.mounts(), key=lambda x: x.mountPoint):
    if m.mountPoint.startswith("/mnt/"):
        print(f"  {m.mountPoint:<25} → {m.source}")